In [4]:
!git clone https://github.com/meta-compute/sem-fracture-deep-learning.git

Cloning into 'sem-fracture-deep-learning'...
remote: Enumerating objects: 2691, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 2691 (delta 0), reused 3 (delta 0), pack-reused 2685 (from 2)
Receiving objects: 100% (2691/2691), 904.02 MiB | 40.42 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Updating files: 100% (2000/2000), done.
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [10]:
cd sem-fracture-deep-learning

/kaggle/working/sem-fracture-deep-learning


In [12]:
!git checkout vishnu-dev
!pip install -r requirements.txt

Updating files: 100% (635/635), done.
branch 'vishnu-dev' set up to track 'origin/vishnu-dev'.
Switched to a new branch 'vishnu-dev'


# Fracture modelling

Use the sections below in order: Train, Validate, then optionally Test. Mixed augmentation is training-only. Final test evaluation is disabled until explicitly enabled. The stored training output is from an earlier interrupted attempt.


## Train

Configure the model in the setup cell and run training. The best validation-macro-F1 checkpoint is saved under `results/shared_pipeline/`.


In [22]:
from pathlib import Path

import pandas as pd
import torch

from sem_fracture.pipeline import run_experiment, run_prediction

DATA_DIR = Path.cwd()
MODEL_KEY = "tinyvit_21m"  # resnet18, deit_tiny, tinyvit_11m, tinyvit_21m
MODE = "finetune"  # head_only or finetune
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

BASELINE_CHECKPOINTS = {
    "resnet18": DATA_DIR / "results" / "resnet18_best.pth",
    "deit_tiny": DATA_DIR / "results" / "deit_tiny" / "checkpoints" / "deit_tiny_best.pth",
    "tinyvit_11m": DATA_DIR / "results" / "tinyvit_11m" / "checkpoints" / "tinyvit_11m_best.pth",
    "tinyvit_21m": DATA_DIR / "results" / "tinyvit_21m" / "checkpoints" / "tinyvit_21m_best.pth",
}

if MODE not in ("head_only", "finetune"):
    raise ValueError(f"Unknown training mode: {MODE}")
if MODEL_KEY not in BASELINE_CHECKPOINTS:
    raise ValueError(f"Unknown model key: {MODEL_KEY}")

INITIAL_CHECKPOINT = BASELINE_CHECKPOINTS[MODEL_KEY] if MODE == "finetune" else None
BACKBONE_LR = 1e-5 if MODE == "finetune" else None
HEAD_LR = 1e-4 if MODE == "finetune" else 1e-3
EPOCHS = 30 if MODEL_KEY == "tinyvit_21m" and MODE == "finetune" else 10
PATIENCE = 4 if MODE == "finetune" else None
GRAD_CLIP_NORM = 1.0 if MODEL_KEY.startswith("tinyvit") and MODE == "finetune" else None
CHECKPOINT_PATH = (
    DATA_DIR / "results" / "shared_pipeline"
    / MODEL_KEY / f"{MODEL_KEY}_{MODE}_best.pth"
)
RUN_FINAL_TEST_EVALUATION = False  # enable only after selecting the checkpoint


In [23]:
result = run_experiment(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    mode=MODE,
    checkpoint_path=CHECKPOINT_PATH,
    device=DEVICE,
    head_lr=HEAD_LR,
    backbone_lr=BACKBONE_LR,
    weight_decay=1e-4,
    epochs=EPOCHS,
    patience=PATIENCE,
    grad_clip_norm=GRAD_CLIP_NORM,
    initial_checkpoint=INITIAL_CHECKPOINT,
    augment_mixed=True,
    seed=42,
    batch_size=16,
)

best = result["best_checkpoint"]
print("Best epoch:", best["epoch"])
print("Best validation macro-F1:", f"{best['val_macro_f1']:.4f}")
print("Checkpoint:", CHECKPOINT_PATH)
print("Training split:", result["split_file"])
print("Class weights:", result["class_weights"])


Epoch 1/30 starting
Epoch 1/30 | train_loss=0.1710 | val_loss=0.1049 | val_accuracy=0.9571 | val_macro_f1=0.9079 | elapsed=30.1s | best_checkpoint=True
Epoch 2/30 starting
Epoch 2/30 | train_loss=0.1115 | val_loss=0.1362 | val_accuracy=0.9464 | val_macro_f1=0.8934 | elapsed=27.5s | best_checkpoint=False
Epoch 3/30 starting
Epoch 3/30 | train_loss=0.0994 | val_loss=0.0727 | val_accuracy=0.9643 | val_macro_f1=0.9213 | elapsed=27.9s | best_checkpoint=True
Epoch 4/30 starting
Epoch 4/30 | train_loss=0.0761 | val_loss=0.0983 | val_accuracy=0.9571 | val_macro_f1=0.9079 | elapsed=27.9s | best_checkpoint=False
Epoch 5/30 starting
Epoch 5/30 | train_loss=0.0665 | val_loss=0.0509 | val_accuracy=0.9786 | val_macro_f1=0.9475 | elapsed=28.0s | best_checkpoint=True
Epoch 6/30 starting
Epoch 6/30 | train_loss=0.0501 | val_loss=0.0692 | val_accuracy=0.9750 | val_macro_f1=0.9420 | elapsed=27.7s | best_checkpoint=False
Epoch 7/30 starting
Epoch 7/30 | train_loss=0.0411 | val_loss=0.0960 | val_accuracy=0

## Validate

Evaluate the selected checkpoint on the original validation split. Validation data is not augmented.


In [24]:
validation_result = run_prediction(
    data_dir=DATA_DIR,
    model_key=MODEL_KEY,
    checkpoint_path=CHECKPOINT_PATH,
    split="validation",
    split_file="dataset_split_augmented.csv",
    device=DEVICE,
    batch_size=16,
)

print(validation_result["metrics"])
pd.DataFrame(validation_result["predictions"])


{'confusion_matrix': [[117, 0, 0], [0, 138, 4], [0, 2, 19]], 'accuracy': 0.9785714285714285, 'precision': [1.0, 0.9857142857142858, 0.8260869565217391], 'recall': [1.0, 0.971830985915493, 0.9047619047619048], 'class_f1': [1.0, 0.9787234042553192, 0.8636363636363636], 'macro_f1': 0.9474532559638943}


,filename,actual,predicted
0,100Cr6_AR_06_10.png,Brittle,Brittle
1,100Cr6_AR_06_3.png,Brittle,Brittle
2,304_H2_02_1.png,Brittle,Brittle
3,304_H2_02_10.png,Brittle,Brittle
4,304_H2_02_11.png,Brittle,Brittle
...,...,...,...
275,mX65_H2_29_22.png,Mixed,Mixed
276,mX65_H2_29_23.png,Mixed,Mixed
277,mX65_H2_29_3.png,Mixed,Mixed
278,mX65_H2_29_31.png,Mixed,Mixed


## Test

After selecting the checkpoint with validation, set `RUN_FINAL_TEST_EVALUATION = True` in the setup cell and run the cell below once. It uses the original `dataset_split.csv`; generated Mixed images belong to training only.


In [25]:
RUN_FINAL_TEST_EVALUATION = True

In [26]:
if RUN_FINAL_TEST_EVALUATION:
    final_test_result = run_prediction(
        data_dir=DATA_DIR,
        model_key=MODEL_KEY,
        checkpoint_path=CHECKPOINT_PATH,
        split="test",
        split_file="dataset_split_augment.csv",
        device=DEVICE,
        batch_size=16,
    )
    print("Final held-out test metrics:", final_test_result["metrics"])
else:
    print("Final test evaluation is disabled.")


Final held-out test metrics: {'confusion_matrix': [[116, 0, 0], [0, 140, 3], [4, 2, 14]], 'accuracy': 0.967741935483871, 'precision': [0.9666666666666667, 0.9859154929577465, 0.8235294117647058], 'recall': [1.0, 0.9790209790209791, 0.7], 'class_f1': [0.983050847457627, 0.9824561403508772, 0.7567567567567567], 'macro_f1': 0.9074212481884203}


## Dual-Attention MIL (TinyViT-21M)

Self-contained configuration, feature caching, DualAttentionMIL model definition, training, validation, and held-out test evaluation.


In [ ]:

from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import csv, json, random, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
import timm

# 1. Reproducibility
SEED = 43

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 2. Project paths
DATA_DIR = Path.cwd()
SPLIT_FILE = DATA_DIR / "dataset_split.csv"

MODEL_NAME = "tiny_vit_21m_224.in1k"

# 3. Existing fine-tuned model
BASELINE_CHECKPOINT = (
    DATA_DIR
    / "results"
    / "tinyvit_21m_finetune"
    / "checkpoints"
    / "tinyvit_21m_finetuned_best.pth"
)

# 4. Separate results for MIL experiment
RESULTS_DIR = (
    DATA_DIR / "results" / "tinyvit_21m_dual_attention_mil"
)

CACHE_DIR = RESULTS_DIR / "cache"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
PLOTS_DIR = RESULTS_DIR / "plots"

for folder in (
    RESULTS_DIR, CACHE_DIR, CHECKPOINT_DIR, PLOTS_DIR
):
    folder.mkdir(parents=True, exist_ok=True)

# 5. Device
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cpu"
)

# 6. Original class mapping
CLASSES = ["Brittle", "Ductile", "Mixed"]

class_to_idx = {
    label: idx for idx, label in enumerate(CLASSES)
}

# 7. Verify required files
assert SPLIT_FILE.is_file(), (
    f"Missing split file: {SPLIT_FILE}"
)

assert BASELINE_CHECKPOINT.is_file(), (
    f"Missing checkpoint: {BASELINE_CHECKPOINT}"
)

# 8. Print configuration
print("Project:", DATA_DIR)
print("Device:", DEVICE)
print("Model:", MODEL_NAME)
print("Source checkpoint:", BASELINE_CHECKPOINT)
print("New results:", RESULTS_DIR)


In [ ]:

# 1. Load the original dataset split
with open(SPLIT_FILE, "r", encoding="utf-8-sig") as file:
    split_records = list(csv.DictReader(file))

# 2. Separate training, validation and test records
train_records = [
    r for r in split_records if r["split"] == "train"
]

val_records = [
    r for r in split_records if r["split"] == "validation"
]

test_records = [
    r for r in split_records if r["split"] == "test"
]

# 3. Verify original dataset sizes
assert (
    len(train_records),
    len(val_records),
    len(test_records)
) == (1402, 280, 279)

# 4. Verify no filename-derived group overlap
train_groups = {r["group"] for r in train_records}
val_groups = {r["group"] for r in val_records}
test_groups = {r["group"] for r in test_records}

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

# 5. Calculate original class weights
train_counts = Counter(
    r["label"] for r in train_records
)

num_samples = len(train_records)
num_classes = len(CLASSES)

class_weights = torch.tensor(
    [
        num_samples / (num_classes * train_counts[label])
        for label in CLASSES
    ],
    dtype=torch.float32,
    device=DEVICE
)

# 6. Display dataset distribution
for name, records in [
    ("Train", train_records),
    ("Validation", val_records),
    ("Test", test_records)
]:
    counts = Counter(r["label"] for r in records)

    mixed_groups = len({
        r["group"]
        for r in records
        if r["label"] == "Mixed"
    })

    print(
        f"{name}: {len(records)} images | "
        f"Class counts: {dict(counts)} | "
        f"Mixed groups: {mixed_groups}"
    )

# 7. Display and verify weights
print("\nCLASS WEIGHTS")
print("-" * 45)

for label, weight in zip(CLASSES, class_weights):
    print(f"{label}: {weight.item():.4f}")

assert torch.allclose(
    class_weights.cpu(),
    torch.tensor([0.7934352, 0.6572902, 4.5816994]),
    rtol=1e-5,
    atol=1e-5
)

print("\nDataset verification successful.")


In [ ]:

# 1. Recreate the original TinyViT-21M architecture
backbone = timm.create_model(
    MODEL_NAME,
    pretrained=False,
    num_classes=3
)

# 2. Load existing fine-tuned checkpoint
source_ckpt = torch.load(
    BASELINE_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

assert source_ckpt.get("model_name", MODEL_NAME) == MODEL_NAME
assert source_ckpt.get("class_to_idx", class_to_idx) == class_to_idx

backbone.load_state_dict(
    source_ckpt["model_state_dict"],
    strict=True
)

# 3. Freeze every parameter
for p in backbone.parameters():
    p.requires_grad = False

backbone = backbone.to(DEVICE)
backbone.eval()

# 4. Verify model parameters
total_params = sum(
    p.numel() for p in backbone.parameters()
)

trainable_params = sum(
    p.numel()
    for p in backbone.parameters()
    if p.requires_grad
)

print("Loaded checkpoint epoch:",
      source_ckpt.get("epoch", "not recorded"))

print("Backbone parameters:", f"{total_params:,}")
print("Trainable parameters:", trainable_params)
print("Classifier input features:", backbone.head.fc.in_features)

assert trainable_params == 0

# 5. Extract features BEFORE the classification layer
@torch.inference_mode()
def extract_embeddings(image_batch):

    spatial = backbone.forward_features(image_batch)

    pooled = backbone.forward_head(
        spatial,
        pre_logits=True
    )

    assert pooled.ndim == 2
    assert pooled.shape[1] == backbone.head.fc.in_features

    return pooled

# 6. Verify feature extraction with a dummy image
dummy = torch.zeros(
    1, 3, 224, 224,
    device=DEVICE
)

embedding = extract_embeddings(dummy)

FEATURE_DIM = embedding.shape[1]

assert FEATURE_DIM == 576

print("\nFEATURE EXTRACTION")
print("-" * 50)
print("Input shape:", tuple(dummy.shape))
print("Embedding shape:", tuple(embedding.shape))
print("Feature dimension:", FEATURE_DIM)
print("Backbone frozen:", trainable_params == 0)
print("\nCell 3 verification successful.")


In [ ]:

# 1. ImageNet normalization — unchanged
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

# 2. Patch configuration
PATCH_SIZE = 224
CANVAS_SIZE = 448

# 50% overlap between adjacent patches
PATCH_STARTS = (0, 112, 224)

# 9 regional crops + 1 global view
N_PATCHES = 10

PATCH_BATCH_SIZE = 16

# Same horizontal/vertical flip types as before
TRAIN_VARIANTS = (
    "none",
    "hflip",
    "vflip",
    "hvflip"
)

# 3. Original normalization
patch_to_tensor = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# 4. Create ten views of one SEM image
def make_bag(image, variant="none"):

    image = image.convert("RGB")

    # Optional flips for training
    if variant in ("hflip", "hvflip"):
        image = ImageOps.mirror(image)

    if variant in ("vflip", "hvflip"):
        image = ImageOps.flip(image)

    # Working image for regional crops
    working = image.resize(
        (CANVAS_SIZE, CANVAS_SIZE),
        Image.Resampling.BILINEAR
    )

    # Extract nine overlapping regions
    crops = [
        working.crop((
            x, y,
            x + PATCH_SIZE,
            y + PATCH_SIZE
        ))
        for y in PATCH_STARTS
        for x in PATCH_STARTS
    ]

    # Add the complete image as the tenth view
    global_view = image.resize(
        (PATCH_SIZE, PATCH_SIZE),
        Image.Resampling.BILINEAR
    )

    crops.append(global_view)

    assert len(crops) == N_PATCHES

    return torch.stack([
        patch_to_tensor(crop)
        for crop in crops
    ], dim=0)


# 5. Test with one original training image
sample_row = train_records[0]

sample_path = (
    DATA_DIR
    / sample_row["label"]
    / sample_row["filename"]
)

assert sample_path.is_file(), (
    f"Missing SEM image: {sample_path}"
)

with Image.open(sample_path) as img:
    sample_bag = make_bag(img, variant="none")

assert sample_bag.shape == (
    10, 3, 224, 224
)

print("Original file:", sample_row["filename"])
print("MIL bag tensor:", tuple(sample_bag.shape))


# 6. Reverse normalization for visualization
def tensor_as_image(x):

    mean = torch.tensor(MEAN).view(3, 1, 1)
    std = torch.tensor(STD).view(3, 1, 1)

    return (
        (x.cpu() * std + mean)
        .clamp(0, 1)
        .permute(1, 2, 0)
        .numpy()
    )


# 7. Plot all ten views
fig, axes = plt.subplots(
    2, 5,
    figsize=(13, 6)
)

for i, ax in enumerate(axes.flat):

    ax.imshow(
        tensor_as_image(sample_bag[i])
    )

    ax.set_title(
        "Global" if i == 9
        else f"Local {i + 1}"
    )

    ax.axis("off")

plt.tight_layout()

plt.savefig(
    PLOTS_DIR / "patch_grid_example.png",
    dpi=160
)

plt.show()


In [ ]:

# 1. Extract and cache features for one dataset split
def cache_one_split_variant(records, split_name, variant):

    out_path = (
        CACHE_DIR
        / f"{split_name}_{variant}_features.pt"
    )

    expected_names = [
        r["filename"] for r in records
    ]

    # Reuse existing cache if already generated
    if out_path.exists():

        data = torch.load(
            out_path,
            map_location="cpu",
            weights_only=True
        )

        assert data["filenames"] == expected_names

        assert data["features"].shape == (
            len(records), N_PATCHES, FEATURE_DIM
        )

        print(f"Reusing {out_path.name}")

        return data["features"].float()

    features = []
    t0 = time.perf_counter()

    backbone.eval()

    # 2. Process every image
    for idx, row in enumerate(records):

        path = (
            DATA_DIR
            / row["label"]
            / row["filename"]
        )

        with Image.open(path) as img:
            bag = make_bag(img, variant)

        # 3. Pass image regions through frozen TinyViT
        patch_vectors = []

        for batch in bag.split(
            PATCH_BATCH_SIZE, dim=0
        ):

            z = extract_embeddings(
                batch.to(DEVICE)
            ).cpu().float()

            patch_vectors.append(z)

        # Shape: (10, 576)
        features.append(
            torch.cat(patch_vectors, dim=0)
        )

        # Progress tracking
        if (
            (idx + 1) % 100 == 0
            or idx + 1 == len(records)
        ):
            print(
                f"{split_name}/{variant}: "
                f"{idx + 1}/{len(records)} images"
            )

    # 4. Combine all image features
    features = torch.stack(
        features, dim=0
    ).contiguous()

    assert features.shape == (
        len(records), N_PATCHES, FEATURE_DIM
    )

    assert torch.isfinite(features).all()

    # 5. Save features to a separate cache
    torch.save(
        {
            "features": features,
            "filenames": expected_names,
            "variant": variant,
            "feature_dim": FEATURE_DIM,
            "patch_count": N_PATCHES,
            "checkpoint_path": str(
                BASELINE_CHECKPOINT
            )
        },
        out_path
    )

    print(
        f"Saved {out_path.name}: "
        f"{tuple(features.shape)}"
    )

    return features


# 6. Training features: same four flip combinations
train_feature_variants = [
    cache_one_split_variant(
        train_records,
        "train",
        variant
    )
    for variant in TRAIN_VARIANTS
]

# 7. Validation features: no augmentation
val_features = cache_one_split_variant(
    val_records,
    "validation",
    "none"
)

# 8. Verify output
print("\nTRAIN FEATURE SHAPES")

for variant, features in zip(
    TRAIN_VARIANTS,
    train_feature_variants
):
    print(variant, tuple(features.shape))

print("\nVALIDATION FEATURE SHAPE")
print(tuple(val_features.shape))

print("\nCell 5 completed successfully.")


In [ ]:

# 1. Convert original labels into two evidence targets
BINARY_TARGETS = torch.tensor([
    [1.0, 0.0],  # Brittle only
    [0.0, 1.0],  # Ductile only
    [1.0, 1.0],  # Mixed: both
], dtype=torch.float32, device=DEVICE)


# 2. Define the new Dual-Attention MIL architecture
class DualAttentionMIL(nn.Module):

    def __init__(self, feature_dim=576, attention_hidden=128):
        super().__init__()

        # Attention branch for brittle features
        self.brittle_attention = nn.Sequential(
            nn.Linear(feature_dim, attention_hidden),
            nn.Tanh(),
            nn.Linear(attention_hidden, 1)
        )

        # Attention branch for ductile features
        self.ductile_attention = nn.Sequential(
            nn.Linear(feature_dim, attention_hidden),
            nn.Tanh(),
            nn.Linear(attention_hidden, 1)
        )

        # Separate evidence prediction heads
        self.brittle_head = nn.Linear(feature_dim, 1)
        self.ductile_head = nn.Linear(feature_dim, 1)

    def forward(self, x):

        # x shape: (batch, 10, 576)

        # 3. Calculate attention weights for each region
        a_b = torch.softmax(
            self.brittle_attention(x).squeeze(-1),
            dim=1
        )

        a_d = torch.softmax(
            self.ductile_attention(x).squeeze(-1),
            dim=1
        )

        # 4. Combine regional features using attention
        b_features = (
            x * a_b.unsqueeze(-1)
        ).sum(dim=1)

        d_features = (
            x * a_d.unsqueeze(-1)
        ).sum(dim=1)

        # 5. Predict evidence for each mechanism
        b_logit = self.brittle_head(b_features)
        d_logit = self.ductile_head(d_features)

        logits = torch.cat(
            [b_logit, d_logit],
            dim=1
        )

        attention = torch.stack(
            [a_b, a_d],
            dim=1
        )

        return logits, attention


# 6. Convert two evidence scores to three-class probabilities
def class_probabilities(binary_logits):

    p = torch.sigmoid(binary_logits)

    pb = p[:, 0]
    pd = p[:, 1]

    joint = torch.stack([
        pb * (1 - pd),       # Brittle only
        (1 - pb) * pd,       # Ductile only
        pb * pd              # Mixed: both
    ], dim=1)

    # Normalize, assuming at least one mechanism is present
    return joint / joint.sum(
        dim=1, keepdim=True
    ).clamp_min(1e-8)


# 7. Initialize the new MIL model
mil_model = DualAttentionMIL(
    feature_dim=FEATURE_DIM,
    attention_hidden=128
).to(DEVICE)

head_trainable = sum(
    p.numel()
    for p in mil_model.parameters()
    if p.requires_grad
)

print("Frozen TinyViT feature width:", FEATURE_DIM)
print(
    "Trainable new MIL-head parameters:",
    f"{head_trainable:,}"
)


# 8. Test the architecture using dummy features
fake_bags = torch.randn(
    2, N_PATCHES, FEATURE_DIM,
    device=DEVICE
)

logits, attention = mil_model(fake_bags)

probs = class_probabilities(logits)


# 9. Verify shapes
assert logits.shape == (2, 2)
assert attention.shape == (2, 2, N_PATCHES)
assert probs.shape == (2, 3)

assert torch.allclose(
    attention.sum(dim=2),
    torch.ones(2, 2, device=DEVICE),
    atol=1e-5
)

assert torch.allclose(
    probs.sum(dim=1),
    torch.ones(2, device=DEVICE),
    atol=1e-5
)

print("\nMODEL VERIFICATION")
print("-" * 50)
print("Input:", tuple(fake_bags.shape))
print("Evidence logits:", tuple(logits.shape))
print("Attention weights:", tuple(attention.shape))
print("Class probabilities:", tuple(probs.shape))
print("\nCell 6 verification successful.")


In [ ]:

# 1. Dataset for cached TinyViT features
class CachedBagDataset(Dataset):

    def __init__(
        self,
        feature_variants,
        records,
        augment=False
    ):

        # Shape: (images, variants, patches, features)
        self.features = torch.stack(
            feature_variants,
            dim=1
        )

        self.labels = torch.tensor(
            [
                class_to_idx[r["label"]]
                for r in records
            ],
            dtype=torch.long
        )

        self.augment = augment

        assert len(self.features) == len(self.labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        # Randomly choose one of the four flip variants
        if self.augment:
            k = torch.randint(
                self.features.shape[1],
                (1,)
            ).item()
        else:
            k = 0

        return (
            self.features[idx, k],
            self.labels[idx]
        )


# 2. Create training and validation datasets
train_dataset = CachedBagDataset(
    train_feature_variants,
    train_records,
    augment=True
)

val_dataset = CachedBagDataset(
    [val_features],
    val_records,
    augment=False
)


# 3. DataLoaders — original batch size
BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)


# 4. Class-weighted binary cross-entropy
def weighted_binary_loss(logits, labels):

    # Convert original labels to two evidence targets
    targets = BINARY_TARGETS[labels]

    # Calculate binary loss for both mechanisms
    two_losses = F.binary_cross_entropy_with_logits(
        logits,
        targets,
        reduction="none"
    ).mean(dim=1)

    # Preserve original image-level class weights
    weights = class_weights[labels]

    return (
        (two_losses * weights).sum()
        / weights.sum()
    )


# 5. Reinitialize MIL head with a fixed seed
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

mil_model = DualAttentionMIL(
    feature_dim=FEATURE_DIM,
    attention_hidden=128
).to(DEVICE)


# 6. Original classifier-head learning parameters
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

MAX_EPOCHS = 30
PATIENCE = 4

optimizer = torch.optim.AdamW(
    mil_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

checkpoint_path = (
    CHECKPOINT_DIR
    / "tinyvit21m_dual_attention_mil_best.pth"
)


# 7. Diagnostic forward and backward pass
bags, labels = next(iter(train_loader))

bags = bags.to(DEVICE)
labels = labels.to(DEVICE)

optimizer.zero_grad(set_to_none=True)

logits, attention = mil_model(bags)

loss = weighted_binary_loss(
    logits,
    labels
)

loss.backward()


# 8. Verify all trainable parameters receive gradients
assert all(
    p.grad is not None
    and torch.isfinite(p.grad).all()
    for p in mil_model.parameters()
)

# No optimizer.step(): parameters remain unchanged
optimizer.zero_grad(set_to_none=True)


# 9. Print verification
print("TRAINING CONFIGURATION")
print("-" * 55)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

print("Batch size:", BATCH_SIZE)
print("Learning rate:", LEARNING_RATE)
print("Weight decay:", WEIGHT_DECAY)

print("Maximum epochs:", MAX_EPOCHS)
print("Early stopping patience:", PATIENCE)

print("\nDIAGNOSTIC")
print("-" * 55)

print("Input bags:", tuple(bags.shape))
print("Evidence logits:", tuple(logits.shape))
print("Loss:", round(loss.item(), 5))

print(
    "Trainable parameters:",
    f"{sum(p.numel() for p in mil_model.parameters()):,}"
)

print("\nCell 7 verification successful.")


In [ ]:

import time
from sklearn.metrics import precision_score

# 1. Initialize training history
history = {
    "train_loss": [],
    "val_loss": [],
    "val_accuracy": [],
    "val_macro_f1": [],
    "val_mixed_precision": [],
    "val_mixed_recall": [],
    "val_mixed_f1": [],
    "epoch_time": []
}

best_f1 = -1.0
best_epoch = 0
epochs_without_improvement = 0

print("TRAINING TINYVIT-21M — DUAL ATTENTION MIL")
print("=" * 85)

print("Architecture:", MODEL_NAME)
print("Device:", DEVICE)
print("Frozen backbone: True")
print("Trainable parameters:", f"{head_trainable:,}")
print("Maximum epochs:", MAX_EPOCHS)
print("Selection metric: Validation macro-F1")
print()

# 2. Training loop
for epoch in range(MAX_EPOCHS):

    if DEVICE.type == "mps":
        torch.mps.synchronize()

    start_time = time.perf_counter()

    mil_model.train()

    total_train_loss = 0.0
    train_samples = 0

    # 3. Training batches
    for bags, labels in train_loader:

        bags = bags.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        logits, attention = mil_model(bags)

        loss = weighted_binary_loss(
            logits, labels
        )

        assert torch.isfinite(loss).item()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            mil_model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        batch_size = labels.size(0)

        total_train_loss += (
            loss.item() * batch_size
        )

        train_samples += batch_size

    train_loss = total_train_loss / train_samples

    # 4. Validation
    mil_model.eval()

    total_val_loss = 0.0
    val_samples = 0

    all_labels = []
    all_predictions = []

    with torch.inference_mode():

        for bags, labels in val_loader:

            bags = bags.to(DEVICE)
            labels_device = labels.to(DEVICE)

            logits, attention = mil_model(bags)

            loss = weighted_binary_loss(
                logits, labels_device
            )

            probabilities = class_probabilities(
                logits
            )

            predictions = probabilities.argmax(dim=1)

            batch_size = labels.size(0)

            total_val_loss += (
                loss.item() * batch_size
            )

            val_samples += batch_size

            all_labels.extend(
                labels.tolist()
            )

            all_predictions.extend(
                predictions.cpu().tolist()
            )

    val_loss = total_val_loss / val_samples

    # 5. Calculate validation metrics
    val_accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    val_macro_f1 = f1_score(
        all_labels,
        all_predictions,
        labels=[0, 1, 2],
        average="macro",
        zero_division=0
    )

    mixed_precision = precision_score(
        all_labels,
        all_predictions,
        labels=[2],
        average="macro",
        zero_division=0
    )

    mixed_recall = recall_score(
        all_labels,
        all_predictions,
        labels=[2],
        average="macro",
        zero_division=0
    )

    mixed_f1 = f1_score(
        all_labels,
        all_predictions,
        labels=[2],
        average="macro",
        zero_division=0
    )

    # 6. Record epoch duration
    if DEVICE.type == "mps":
        torch.mps.synchronize()

    epoch_time = time.perf_counter() - start_time

    # 7. Store training history
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    history["val_accuracy"].append(float(val_accuracy))
    history["val_macro_f1"].append(float(val_macro_f1))
    history["val_mixed_precision"].append(float(mixed_precision))
    history["val_mixed_recall"].append(float(mixed_recall))
    history["val_mixed_f1"].append(float(mixed_f1))
    history["epoch_time"].append(float(epoch_time))

    # 8. Save best model based on validation macro-F1
    improved = val_macro_f1 > best_f1 + 1e-6

    if improved:

        best_f1 = val_macro_f1
        best_epoch = epoch + 1

        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": best_epoch,
                "model_state_dict": {
                    k: v.detach().cpu().clone()
                    for k, v in mil_model.state_dict().items()
                },
                "model_name": MODEL_NAME,
                "feature_dim": FEATURE_DIM,
                "attention_hidden": 128,
                "n_patches": N_PATCHES,
                "val_macro_f1": float(val_macro_f1),
                "val_accuracy": float(val_accuracy),
                "val_mixed_f1": float(mixed_f1),
                "val_mixed_recall": float(mixed_recall),
                "source_checkpoint": str(
                    BASELINE_CHECKPOINT
                )
            },
            checkpoint_path
        )

    else:
        epochs_without_improvement += 1

    # 9. Print epoch metrics
    print(
        f"Epoch {epoch + 1:02d}/{MAX_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Accuracy: {val_accuracy:.4f} | "
        f"Macro-F1: {val_macro_f1:.4f} | "
        f"Mixed P/R/F1: "
        f"{mixed_precision:.4f}/"
        f"{mixed_recall:.4f}/"
        f"{mixed_f1:.4f} | "
        f"Time: {epoch_time:.1f}s"
        + (" | BEST" if improved else "")
    )

    # 10. Early stopping
    if epochs_without_improvement >= PATIENCE:

        print(
            f"\nEarly stopping after epoch {epoch + 1}"
        )

        break


# 11. Restore best checkpoint
best_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

mil_model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

mil_model = mil_model.to(DEVICE)
mil_model.eval()

assert best_checkpoint["epoch"] == best_epoch

# 12. Save training history
history_path = RESULTS_DIR / "training_history.json"

with open(history_path, "w") as f:
    json.dump(history, f, indent=4)

# 13. Final summary
print("\nTRAINING COMPLETE")
print("=" * 65)

print("Best epoch:", best_epoch)
print(f"Best validation macro-F1: {best_f1:.4f}")

print(
    "Best validation Mixed F1:",
    f"{best_checkpoint['val_mixed_f1']:.4f}"
)

print("Saved checkpoint:", checkpoint_path)
print("Saved history:", history_path)

print("\nCell 8 completed successfully.")


In [ ]:

# 1. Load best MIL checkpoint
best_checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

mil_model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

mil_model.eval()
backbone.eval()

print("BEST MIL CHECKPOINT")
print("-" * 60)
print("Epoch:", best_checkpoint["epoch"])
print("Validation Macro-F1:",
      best_checkpoint["val_macro_f1"])


# 2. MIL validation predictions
y_true = []
mil_preds = []
mil_probs = []

with torch.inference_mode():

    for bags, labels in val_loader:

        logits, attention = mil_model(
            bags.to(DEVICE)
        )

        probs = class_probabilities(logits)

        y_true.extend(labels.tolist())
        mil_preds.extend(
            probs.argmax(dim=1).cpu().tolist()
        )

        mil_probs.extend(
            probs.cpu().tolist()
        )


# 3. Original TinyViT predictions
# The 10th cached view is the complete image.
# Its 576 features feed the original classifier.

original_preds = []

with torch.inference_mode():

    for start in range(
        0, len(val_features), BATCH_SIZE
    ):

        global_features = val_features[
            start:start + BATCH_SIZE, -1, :
        ].to(DEVICE)

        logits = backbone.head.fc(
            global_features
        )

        original_preds.extend(
            logits.argmax(dim=1).cpu().tolist()
        )


# 4. Verify alignment and original results
assert len(y_true) == 280
assert len(original_preds) == 280
assert len(mil_preds) == 280

assert y_true == [
    class_to_idx[r["label"]]
    for r in val_records
]

original_cm = confusion_matrix(
    y_true, original_preds, labels=[0, 1, 2]
)

mil_cm = confusion_matrix(
    y_true, mil_preds, labels=[0, 1, 2]
)

# Original fine-tuned checkpoint should reproduce
# the previously reported confusion matrix.
expected_original_cm = np.array([
    [117, 0, 0],
    [0, 142, 0],
    [0, 2, 19]
])

assert np.array_equal(
    original_cm, expected_original_cm
), (
    "Original predictions differ from the prior "
    "validation results. Verify cached global features."
)


# 5. Compare evaluation metrics
print("\nVALIDATION COMPARISON")
print("=" * 70)

for name, predictions in [
    ("Original TinyViT-21M", original_preds),
    ("Dual-Attention MIL", mil_preds)
]:

    acc = accuracy_score(y_true, predictions)

    macro = f1_score(
        y_true, predictions,
        average="macro",
        zero_division=0
    )

    mixed_f1 = f1_score(
        y_true, predictions,
        labels=[2],
        average="macro",
        zero_division=0
    )

    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro-F1: {macro:.4f}")
    print(f"Mixed F1: {mixed_f1:.4f}")

    print(classification_report(
        y_true, predictions,
        labels=[0, 1, 2],
        target_names=CLASSES,
        digits=4,
        zero_division=0
    ))


# 6. Prediction-level audit
audit_rows = []

for i, record in enumerate(val_records):

    actual = y_true[i]
    old = original_preds[i]
    new = mil_preds[i]

    if old != actual and new == actual:
        status = "Corrected"
    elif old == actual and new != actual:
        status = "New error"
    elif old != actual and new != actual:
        status = "Still wrong"
    else:
        status = "Correct"

    audit_rows.append({
        "filename": record["filename"],
        "group": record["group"],
        "actual": CLASSES[actual],
        "original": CLASSES[old],
        "mil": CLASSES[new],
        "status": status,
        "mil_confidence": max(mil_probs[i])
    })

audit = pd.DataFrame(audit_rows)

changed = audit[
    audit["status"] != "Correct"
].copy()

print("\nPREDICTION AUDIT")
print("=" * 70)

print(audit["status"].value_counts())

print("\nCORRECTED / MISCLASSIFIED IMAGES")
print(changed.to_string(index=False))


# 7. Save audit
audit_path = (
    RESULTS_DIR / "validation_prediction_audit.csv"
)

audit.to_csv(audit_path, index=False)


# 8. Plot confusion matrices and learning curves
fig, axes = plt.subplots(
    2, 2,
    figsize=(13, 10)
)

ConfusionMatrixDisplay(
    original_cm,
    display_labels=CLASSES
).plot(
    ax=axes[0, 0],
    cmap="Blues",
    colorbar=False,
    values_format="d"
)

axes[0, 0].set_title("Original Fine-Tuned TinyViT-21M")

ConfusionMatrixDisplay(
    mil_cm,
    display_labels=CLASSES
).plot(
    ax=axes[0, 1],
    cmap="Blues",
    colorbar=False,
    values_format="d"
)

axes[0, 1].set_title("Dual-Attention MIL")

epochs = np.arange(
    1, len(history["train_loss"]) + 1
)

axes[1, 0].plot(
    epochs, history["train_loss"],
    label="Training Loss"
)

axes[1, 0].plot(
    epochs, history["val_loss"],
    label="Validation Loss"
)

axes[1, 0].set_title("Training and Validation Loss")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(
    epochs, history["val_macro_f1"],
    label="Macro-F1"
)

axes[1, 1].plot(
    epochs, history["val_mixed_f1"],
    label="Mixed F1"
)

axes[1, 1].axvline(
    best_checkpoint["epoch"],
    linestyle="--",
    label="Best Epoch"
)

axes[1, 1].set_title("Validation F1 Scores")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()

plot_path = (
    PLOTS_DIR / "mil_vs_original_validation.png"
)

plt.savefig(plot_path, dpi=160)
plt.show()

print("\nSaved audit:", audit_path)
print("Saved plot:", plot_path)
print("\nCell 9 completed successfully.")


In [ ]:

# CELL 10 — HELD-OUT TEST EVALUATION

# 1. Extract and cache test image features
# No augmentation and no training
test_features = cache_one_split_variant(
    test_records,
    "test",
    "none"
)

# 2. Create test dataset
test_dataset = CachedBagDataset(
    [test_features],
    test_records,
    augment=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

assert len(test_dataset) == 279

# 3. Freeze both models for evaluation
backbone.eval()
mil_model.eval()

test_true = []
original_test_preds = []
mil_test_preds = []
mil_test_probs = []

# 4. Evaluate both models on identical images
with torch.inference_mode():

    for bags, labels in test_loader:

        bags = bags.to(DEVICE)

        # Original TinyViT:
        # Use the global (10th) image view
        global_features = bags[:, -1, :]

        original_logits = backbone.head.fc(
            global_features
        )

        original_predictions = (
            original_logits.argmax(dim=1)
        )

        # Dual-Attention MIL:
        # Use all 10 image views
        mil_logits, attention = mil_model(bags)

        probabilities = class_probabilities(
            mil_logits
        )

        mil_predictions = probabilities.argmax(dim=1)

        test_true.extend(labels.tolist())

        original_test_preds.extend(
            original_predictions.cpu().tolist()
        )

        mil_test_preds.extend(
            mil_predictions.cpu().tolist()
        )

        mil_test_probs.extend(
            probabilities.cpu().tolist()
        )

# 5. Verify alignment
assert len(test_true) == 279

assert test_true == [
    class_to_idx[r["label"]]
    for r in test_records
]

# 6. Confirm original TinyViT test predictions
original_test_cm = confusion_matrix(
    test_true,
    original_test_preds,
    labels=[0, 1, 2]
)

expected_original_test_cm = np.array([
    [115, 0, 1],
    [0, 139, 4],
    [3, 3, 14]
])

assert np.array_equal(
    original_test_cm,
    expected_original_test_cm
), "Original TinyViT test results did not reproduce."

# 7. Calculate MIL confusion matrix
mil_test_cm = confusion_matrix(
    test_true,
    mil_test_preds,
    labels=[0, 1, 2]
)

# 8. Compare both models
test_results = {}

print("TINYVIT-21M — TEST COMPARISON")
print("=" * 70)

for name, predictions in [
    ("Original TinyViT", original_test_preds),
    ("Dual-Attention MIL", mil_test_preds)
]:

    acc = accuracy_score(test_true, predictions)

    macro = f1_score(
        test_true, predictions,
        average="macro",
        zero_division=0
    )

    mixed_f1 = f1_score(
        test_true, predictions,
        labels=[2],
        average="macro",
        zero_division=0
    )

    print(f"\n{name}")
    print("-" * 60)
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro-F1: {macro:.4f}")
    print(f"Mixed F1: {mixed_f1:.4f}")

    report = classification_report(
        test_true,
        predictions,
        labels=[0, 1, 2],
        target_names=CLASSES,
        digits=4,
        output_dict=True,
        zero_division=0
    )

    print(classification_report(
        test_true,
        predictions,
        labels=[0, 1, 2],
        target_names=CLASSES,
        digits=4,
        zero_division=0
    ))

    test_results[name] = {
        "accuracy": float(acc),
        "macro_f1": float(macro),
        "mixed_f1": float(mixed_f1),
        "classification_report": report
    }

# 9. Identify corrected and new errors
audit_rows = []

for i, record in enumerate(test_records):

    actual = test_true[i]
    old = original_test_preds[i]
    new = mil_test_preds[i]

    if old != actual and new == actual:
        status = "Corrected"
    elif old == actual and new != actual:
        status = "New error"
    elif old != actual and new != actual:
        status = "Still wrong"
    else:
        status = "Correct"

    audit_rows.append({
        "filename": record["filename"],
        "group": record["group"],
        "actual": CLASSES[actual],
        "original": CLASSES[old],
        "mil": CLASSES[new],
        "status": status,
        "mil_confidence": max(mil_test_probs[i])
    })

test_audit = pd.DataFrame(audit_rows)

print("\nTEST PREDICTION AUDIT")
print("=" * 70)

print(test_audit["status"].value_counts())

print("\nMISCLASSIFIED / CORRECTED IMAGES")

print(
    test_audit[
        test_audit["status"] != "Correct"
    ].to_string(index=False)
)

# 10. Save test results
test_results_path = (
    RESULTS_DIR / "test_comparison.json"
)

test_audit_path = (
    RESULTS_DIR / "test_prediction_audit.csv"
)

with open(test_results_path, "w") as f:
    json.dump(test_results, f, indent=4)

test_audit.to_csv(
    test_audit_path,
    index=False
)

print("\nSaved:", test_results_path)
print("Saved:", test_audit_path)
print("\nCell 10 completed successfully.")
